# Benchmark Minerals Intelligence
## Precious Metals & Critical Minerals Dashboard

Interactive data exploration for daily pricing, market indices, and AI-powered analyst research.

---
## 1. Available Data

In [ ]:
SHOW TABLES IN PUBLIC;

In [ ]:
SHOW SEMANTIC VIEWS IN PUBLIC;

---
## 2. Precious Metals Pricing Overview

In [ ]:
SELECT MATERIAL, EXCHANGE,
       MAX(ASSESSMENT_DATE) AS LATEST_DATE,
       MAX_BY(PRICE_CLOSE_USD_PER_OZ, ASSESSMENT_DATE) AS LATEST_CLOSE_USD_OZ,
       MAX_BY(MONTH_OVER_MONTH_CHANGE_PCT, ASSESSMENT_DATE) AS MOM_CHANGE_PCT,
       MAX_BY(YEAR_OVER_YEAR_CHANGE_PCT, ASSESSMENT_DATE) AS YOY_CHANGE_PCT
FROM PUBLIC.BATTERY_CRITICAL_MINERALS_PRICING
WHERE PRICE_CLOSE_USD_PER_OZ IS NOT NULL
GROUP BY MATERIAL, EXCHANGE
ORDER BY MATERIAL, EXCHANGE;

---
## 3. Price Trend Chart

In [ ]:
import streamlit as st
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
    SELECT ASSESSMENT_DATE, MATERIAL, EXCHANGE, PRICE_CLOSE_USD_PER_OZ
    FROM PUBLIC.BATTERY_CRITICAL_MINERALS_PRICING
    WHERE PRICE_CLOSE_USD_PER_OZ IS NOT NULL
    ORDER BY ASSESSMENT_DATE
""").to_pandas()

chart = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x=alt.X("ASSESSMENT_DATE:T", title="Date"),
        y=alt.Y("PRICE_CLOSE_USD_PER_OZ:Q", title="Close Price (USD/oz)"),
        color=alt.Color("MATERIAL:N", title="Metal"),
        strokeDash=alt.StrokeDash("EXCHANGE:N", title="Exchange"),
        tooltip=[
            alt.Tooltip("ASSESSMENT_DATE:T", title="Date"),
            alt.Tooltip("MATERIAL:N", title="Metal"),
            alt.Tooltip("EXCHANGE:N", title="Exchange"),
            alt.Tooltip("PRICE_CLOSE_USD_PER_OZ:Q", title="Close", format=",.2f"),
        ],
    )
    .properties(width=900, height=400)
    .interactive()
)
st.altair_chart(chart, use_container_width=True)

---
## 4. Daily Price Changes

In [ ]:
WITH ranked AS (
    SELECT MATERIAL, ASSESSMENT_DATE, AVG(PRICE_CLOSE_USD_PER_OZ) AS AVG_CLOSE,
           ROW_NUMBER() OVER (PARTITION BY MATERIAL ORDER BY ASSESSMENT_DATE DESC) AS rn
    FROM PUBLIC.BATTERY_CRITICAL_MINERALS_PRICING
    WHERE PRICE_CLOSE_USD_PER_OZ IS NOT NULL
    GROUP BY MATERIAL, ASSESSMENT_DATE
)
SELECT
    c.MATERIAL,
    c.ASSESSMENT_DATE AS LATEST_DATE,
    ROUND(c.AVG_CLOSE, 2) AS LATEST_CLOSE,
    ROUND(c.AVG_CLOSE - p.AVG_CLOSE, 2) AS DAILY_CHANGE,
    ROUND((c.AVG_CLOSE - p.AVG_CLOSE) / NULLIF(p.AVG_CLOSE, 0) * 100, 2) AS CHANGE_PCT
FROM ranked c
JOIN ranked p ON c.MATERIAL = p.MATERIAL AND c.rn = 1 AND p.rn = 2
ORDER BY CHANGE_PCT DESC;

---
## 5. Critical Minerals (per kg / per tonne)

In [ ]:
SELECT MATERIAL, MATERIAL_GRADE, REGION,
       MAX(ASSESSMENT_DATE) AS LATEST_DATE,
       MAX_BY(PRICE_USD_PER_KG, ASSESSMENT_DATE) AS PRICE_PER_KG,
       MAX_BY(PRICE_USD_PER_TONNE, ASSESSMENT_DATE) AS PRICE_PER_TONNE,
       MAX_BY(TREND_PCT, ASSESSMENT_DATE) AS TREND_PCT,
       MAX_BY(SUPPLY_RISK, ASSESSMENT_DATE) AS SUPPLY_RISK
FROM PUBLIC.BATTERY_CRITICAL_MINERALS_PRICING
WHERE PRICE_USD_PER_KG IS NOT NULL
GROUP BY MATERIAL, MATERIAL_GRADE, REGION
ORDER BY MATERIAL, MATERIAL_GRADE;

---
## 6. Market Indices

In [ ]:
SELECT INDEX_NAME,
       MAX(INDEX_DATE) AS LATEST_DATE,
       MAX_BY(INDEX_VALUE, INDEX_DATE) AS LATEST_VALUE,
       MAX_BY(WEEKLY_CHANGE_PCT, INDEX_DATE) AS WEEKLY_CHG,
       MAX_BY(MONTHLY_CHANGE_PCT, INDEX_DATE) AS MONTHLY_CHG,
       MAX_BY(YTD_CHANGE_PCT, INDEX_DATE) AS YTD_CHG
FROM PUBLIC.MARKET_INDICES
GROUP BY INDEX_NAME
ORDER BY LATEST_VALUE DESC;

In [ ]:
import streamlit as st
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

idx_df = session.sql("""
    SELECT INDEX_DATE, INDEX_NAME, INDEX_VALUE
    FROM PUBLIC.MARKET_INDICES
    ORDER BY INDEX_DATE
""").to_pandas()

chart = (
    alt.Chart(idx_df)
    .mark_line()
    .encode(
        x=alt.X("INDEX_DATE:T", title="Date"),
        y=alt.Y("INDEX_VALUE:Q", title="Index Value (Base 100)"),
        color=alt.Color("INDEX_NAME:N", title="Index"),
        tooltip=[
            alt.Tooltip("INDEX_DATE:T", title="Date"),
            alt.Tooltip("INDEX_NAME:N", title="Index"),
            alt.Tooltip("INDEX_VALUE:Q", title="Value", format=",.2f"),
        ],
    )
    .properties(width=900, height=400)
    .interactive()
)
st.altair_chart(chart, use_container_width=True)

---
## 7. Battery Supply Chain

In [ ]:
SELECT MINERAL, COUNT(DISTINCT MINE_NAME) AS MINES,
       COUNT(DISTINCT COUNTRY) AS COUNTRIES,
       SUM(ANNUAL_PRODUCTION_TONNES) AS TOTAL_PRODUCTION,
       AVG(ESG_RISK_SCORE) AS AVG_ESG_RISK
FROM PUBLIC.MINING_OPERATIONS
GROUP BY MINERAL
ORDER BY TOTAL_PRODUCTION DESC;

In [ ]:
SELECT MINERAL, YEAR, SUPPLY_TONNES, DEMAND_TONNES,
       ROUND(DEMAND_TONNES - SUPPLY_TONNES) AS DEFICIT,
       DEMAND_GROWTH_PCT
FROM PUBLIC.SUPPLY_DEMAND_FORECASTS
ORDER BY MINERAL, YEAR;

In [ ]:
SELECT COUNTRY, CHEMISTRY_TYPE,
       COUNT(*) AS MANUFACTURERS,
       SUM(CAPACITY_GWH) AS TOTAL_GWH
FROM PUBLIC.BATTERY_CELL_MANUFACTURERS
GROUP BY COUNTRY, CHEMISTRY_TYPE
ORDER BY TOTAL_GWH DESC
LIMIT 20;

---
## 8. Analyst Reports

In [ ]:
SELECT REPORT_NAME, COUNT(*) AS CHUNKS
FROM PUBLIC.ANALYST_REPORT_CHUNKS
GROUP BY REPORT_NAME
ORDER BY REPORT_NAME;

---
## 9. Benchmark Minerals Agent

Use the Cortex Agent via **Snowflake Intelligence** for natural language questions.

**Sample questions:**
- What is the latest gold price?
- Compare lithium and cobalt prices over the last 6 months
- Which metal had the biggest price increase this year?
- What is Goldman Sachs gold price forecast for 2026?
- What are the latest values for all market indices?

---
## 10. Custom Query

In [ ]:
-- Write your custom SQL here
SELECT * FROM PUBLIC.BATTERY_CRITICAL_MINERALS_PRICING LIMIT 10;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.sql("SELECT * FROM PUBLIC.BATTERY_CRITICAL_MINERALS_PRICING LIMIT 10").to_pandas()
df